# 01 - Import et nettoyage des données
## Ecommerce_UK_Retail

Dataset source : Online Retail (transactions e-commerce UK, 2010-2011).

In [5]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_excel("../data/raw/Online Retail.xlsx")
print("Shape brut :", df.shape)
df.head()

Shape brut : (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [7]:
# structure et type
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [8]:
# valeurs manquantes
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

**Remarque méthodologique** : la suppression des `CustomerID` manquants est un choix
adapté à l'objectif de ce projet, centré sur l'analyse par client (panier moyen,
clients uniques, comportement d'achat). Ce choix n'est pas universel - il dépend de
la question business posée : si l'objectif avait été de calculer le chiffre d'affaires
total réel de l'entreprise (achats invités inclus), il aurait fallu le calculer
**avant** cette suppression, sur les données brutes, pour ne pas sous-estimer le CA
d'environ 33% (135 080 lignes concernées sur 541 909).

In [9]:
df = df.dropna(subset=["CustomerID"])
print("Shape après suppression CustomerID manquant :", df.shape)

Shape après suppression CustomerID manquant : (406829, 8)


In [12]:
# suppression des commandes annulées
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
annulations = df["InvoiceNo"].str.startswith("C")
print(f"{annulations.sum()} lignes annulées détectées")
df = df[~annulations]

8905 lignes annulées détectées


In [13]:
# suppression des quantités/prix incohérents
avant = len(df)
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
print(f"{avant - len(df)} lignes supprimées (quantité/prix <= 0)")

40 lignes supprimées (quantité/prix <= 0)


In [14]:
# suppression des doublons
doublons = df.duplicated().sum()
print(f"{doublons} lignes dupliquées détectées")
df = df.drop_duplicates()

5192 lignes dupliquées détectées


In [15]:
# correction des types
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["CustomerID"] = df["CustomerID"].astype(int)
df.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID              int64
Country                object
dtype: object

In [16]:
print("Shape final:", df.shape)
print("Valeurs manquantes restantes :\n", df.isnull().sum())
df.describe()

Shape final: (392692, 8)
Valeurs manquantes restantes :
 InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


,Quantity,InvoiceDate,UnitPrice,CustomerID
count,392692.000000,392692,392692.000000,392692.000000
mean,13.119702,2011-07-10 19:13:07.771892480,3.125914,15287.843865
min,1.000000,2010-12-01 08:26:00,0.001000,12346.000000
25%,2.000000,2011-04-07 11:12:00,1.250000,13955.000000
50%,6.000000,2011-07-31 12:02:00,1.950000,15150.000000
75%,12.000000,2011-10-20 12:53:00,3.750000,16791.000000
max,80995.000000,2011-12-09 12:50:00,8142.750000,18287.000000
std,180.492832,NaN,22.241836,1713.539549


In [17]:
df.to_csv("../data/processed/OnlineRetail_clean.csv", index=False)
print("Sauvegardé dans data/processed/OnlineRetail_clean.csv —", df.shape)

Sauvegardé dans data/processed/OnlineRetail_clean.csv — (392692, 8)
